In [1]:
import sys
import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

class NavierStokesSolver:
    def __init__(self, mms_type=1, reynolds=400.0):
        self.mms_type = mms_type
        self.re = reynolds
        self.dm = None
        self.ts = None
        self.F = None
        self.J = None
        self.P = None

    def create_mesh(self):
        self.dm = PETSc.DMPlex().createBoxMesh(
            faces=[4, 4], 
            lower=[0, 0], 
            upper=[1, 1], 
            simplex=True, 
            interpolate=True
        )
        self.dm.setFromOptions()
        self.dm.viewFromOptions("-dm_view")

    def setup_discretization(self):
        """Sets up Finite Element spaces for velocity and pressure."""
        # P2 elements for velocity, P1 for pressure (Taylor-Hood)
        fe_vel = PETSc.FE().createDefault(self.dm.getDimension(), self.dm.getDimension(), 
                                         True, "vel_", PETSC_DEFAULT)
        fe_pres = PETSc.FE().createDefault(self.dm.getDimension(), 1, 
                                          True, "pres_", PETSC_DEFAULT)
        
        self.dm.setField(0, fe_vel)
        self.dm.setField(1, fe_pres)
        self.dm.createDS()

        ds = self.dm.getDS()

        # You must define Python callbacks matching PetscDS pointwise signatures
        ds.setResidual(0, self.f0_mms1_u, self.f1_u)
        ds.setResidual(1, self.f0_p, self.f1_p)

        ds.setJacobian(0, 0, self.g0_uu, self.g1_uu, None, self.g3_uu)
        ds.setJacobian(0, 1, None, None, self.g2_up, None)
        ds.setJacobian(1, 0, None, self.g1_pu, None, None)

        ds.setExactSolution(0, self.mms1_u_2d)
        ds.setExactSolution(1, self.mms1_p_2d)

        # Add essential velocity boundary condition on marker=1
        # Exact signature depends on your petsc4py version
        # dm.addBoundary(...)

        self.dm.createClosureIndex(None)

        # Pressure nullspace like C example
        pressure_disc = self.dm.getField(1)[1]
        nsp = PETSc.NullSpace().create(constant=True, comm=PETSc.COMM_WORLD)
        pressure_disc.compose("nullspace", nsp)

    def setup_ts(self):
        self.ts = PETSc.TS().create(PETSc.COMM_WORLD)
        self.ts.setDM(self.dm)
        self.ts.setProblemType(PETSc.TS.ProblemType.NONLINEAR)
        self.ts.setExactFinalTime(PETSc.TS.ExactFinalTime.STEPOVER)

        # If your petsc4py exposes DM-local TS hooks, use those.
        # That is the closest match to ex46.c.
        #
        # self.dm.setTSBoundaryLocal(PETSc.DMPlex.computeBoundary, self)
        # self.dm.setTSIFunctionLocal(PETSc.DMPlex.computeIFunctionFEM, self)
        # self.dm.setTSIJacobianLocal(PETSc.DMPlex.computeIJacobianFEM, self)
        #
        # Otherwise fall back to TS-level wrappers:

        self.F = self.dm.createGlobalVec()
        self.J = self.dm.createMat()
        self.P = self.J

        self.ts.setIFunction(self.formIFunction, self.F)
        self.ts.setIJacobian(self.formIJacobian, self.J, self.P)

        self.ts.setFromOptions()

    def initialize_solution(self):
        u = self.dm.createGlobalVec()
        ds = self.dm.getDS()
        funcs = [ds.getExactSolution(0)[0], ds.getExactSolution(1)[0]]
        ctxs  = [ds.getExactSolution(0)[1], ds.getExactSolution(1)[1]]

        # Exact bound signature depends on your petsc4py build
        self.dm.projectFunction(0.0, funcs, ctxs, PETSc.InsertMode.INSERT_ALL_VALUES, u)
        return u

    def formIFunction(self, ts, t, U, Udot, F):
        # If direct DMPlex FEM assembler is exposed in your build, call it here.
        # The exact callable name may differ by petsc4py version.
        #
        # PETSc.DMPlex.computeIFunctionFEM(self.dm, t, U, Udot, F, self)
        #
        # or possibly:
        # self.dm.computeIFunctionFEM(t, U, Udot, F, self)
        raise NotImplementedError

    def formIJacobian(self, ts, t, U, Udot, shift, J, P):
        # Same idea as formIFunction
        raise NotImplementedError

TypeError: createBoxMesh() got an unexpected keyword argument 'dim'